# Проект 1. Окупилось ли повышение цен в марте 2026

**Бизнес-вопрос.** С 1 марта 2026 ПРАЙМ поднял цены: "Базовый" с 199 до 249 руб., "Расширенный" с 399 до 449 руб.
Принесло ли это больше денег, или часть выигрыша съел отток? Что делать с ценами дальше?

**Гипотезы**, которые проверяем в ноутбуке:

1. Выручка после повышения выросла за счёт цены, а не за счёт базы подписок.
2. После повышения вырос отток: клиенты ушли из-за цены.
3. Сильнее ушли клиенты дешёвого тарифа или отдельных каналов привлечения.

**Договорённости расчёта** (пишем их в подписи к каждому числу):

- время и месяцы считаем по **UTC**, так хранит база;
- выручка это сумма **успешных платежей** (статус `status = 'success'`), возвраты (`refunded`) и неудачные (`failed`) не берём;
- выручку берём из таблицы платежей (`payments`) и **не соединяем** с участниками подписок (`subscription_members`): так нет двойного счёта из-за шеринга;
- месяцы разной длины сравниваем **в пересчёте на день**;
- клиентов без канала привлечения показываем отдельной строкой "без канала".

## 0. Подключение

Функция запроса (`q`) отправляет SQL в базу и возвращает таблицу pandas. Пароля в ноутбуке нет:
его хранит файл `.env`, а читает функция подключения (`get_engine`) из `src/prime/config.py`.

In [ ]:
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sqlalchemy import text

# Наши функции из репозитория: подключение, проверки, метрики, разложение.
try:
    from prime.config import get_engine
    from prime.checks import find_duplicate_payments, find_missing_channel
    from prime.metrics import active_on, churn_rate
    from prime.factors import chain, contrib, waterfall
except ModuleNotFoundError:
    # Понятная ошибка вместо длинного трейсбека: ноутбук открыт не в окружении проекта.
    raise ModuleNotFoundError(
        "Пакет prime не найден. Запустите Jupyter из корня репозитория командой: uv run jupyter lab\n"
        f"Сейчас Python отсюда: {sys.executable}"
    )

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить SQL-запрос и вернуть DataFrame. Соединение закрываем сразу."""
    try:
        with engine.connect() as conn:
            # Сутки и месяцы считаем по UTC: так они одинаковые у всех, кто запускает ноутбук.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        # Сервер общий на всю группу, лишние соединения не держим.
        engine.dispose()


# Тысячи через пробел в выводе таблиц: так числа читать легче.
pd.set_option("display.float_format", lambda x: f"{x:,.2f}".replace(",", " "))

print("pandas", pd.__version__)
print("подключились как:", q("select current_user as login").iloc[0, 0])

**Вывод.** Подключение работает: pandas 3.0.5, вошли под своим логином. Функции из `src/prime` импортировались.

**Следующий шаг: приёмка данных.** Ошибки pandas не падают, а тихо меняют числа (Л3). Поэтому до любого расчёта
проверяем таблицы семью вопросами приёмки.

## 1. Приёмка данных

Задаём таблицам семь вопросов приёмки из Л3: сколько строк и сколько ждали, какие типы, сколько весит,
одна строка это кто, где пропуски, нет ли невозможных значений, нет ли дыр по датам.

Нужны три таблицы:

| Таблица | Зерно (одна строка это) | Зачем нам |
|---|---|---|
| справочник тарифов (`tariffs`) | цена тарифа в интервале дат | узнать, когда и насколько подняли цены |
| платежи (`payments`) | одна попытка списания | выручка и число платежей |
| подписки (`subscriptions`) | одна подписка | активная база, отток, канал владельца |

### 1.1 Справочник цен

У тарифа несколько строк: цена меняется, и у каждой строки свой интервал от даты начала (`valid_from`)
до даты конца (`valid_to`). На семинаре 4 в нём нашлась строка, которая перекрывает соседние.

In [ ]:
tariffs = q("select * from prime.tariffs order by tariff_id, valid_from")

print(tariffs.dtypes)
tariffs

**Вывод.** В справочнике 5 строк на 2 тарифа. Цена в месяц (`monthly_fee`) приехала числом, а даты начала и конца
(`valid_from`, `valid_to`) приехали объектами (`object`), а не временем: сравнивать их с моментом платежа напрямую нельзя.
Подозрительна строка "Расширенного" за 429 руб. с 01.02.2026 по 30.06.2026: она залезает и на старую, и на новую цену.

**Следующий шаг: найти пересечения интервалов кодом.** На 5 строках это видно глазами, но вывод должен опираться
на проверку, которая сработает на любом справочнике (Л4, слайд 14).

In [ ]:
# Ищем пересечения интервалов самосоединением (Л4, слайд 14).
# Соединяем справочник сам с собой по тарифу: получаем все пары строк одного тарифа.
a = tariffs.reset_index(names="row_id")
pairs = a.merge(a, on="tariff_id", suffixes=("_a", "_b"))

overlap = pairs[
    (pairs["row_id_a"] < pairs["row_id_b"])               # без этого каждая строка "пересечётся" сама с собой
    & (pairs["valid_from_a"] <= pairs["valid_to_b"])      # два интервала пересекаются, если каждый
    & (pairs["valid_from_b"] <= pairs["valid_to_a"])      # начинается не позже конца другого
]

print("пересекающихся пар:", len(overlap))
overlap[["tariff_id", "monthly_fee_a", "valid_from_a", "valid_to_a", "monthly_fee_b", "valid_from_b", "valid_to_b"]]

**Вывод.** Пересекающихся пар 2, и в обеих участвует строка 429 руб.: она перекрывает цену 399 руб. (февраль 2026)
и цену 449 руб. (с марта). С таким справочником каждый платёж "Расширенного" за февраль-июнь найдёт две цены.

**Следующий шаг: загрузить платежи.** Какую строку считать ошибкой, решаем не глазами, а по данным:
в разделе 1.3 посмотрим, какие суммы реально списывались.

### 1.2 Платежи

Берём период с декабря 2025 по июнь 2026: три месяца до повышения, март и три месяца после.
Июль и август не берём: в конце августа случился обвал партнёрского канала (`partner_telecom`, Л6), он смешал бы два разных события.

Колонки выбираем в SQL, а не "выгружу всё, потом посмотрю": так таблица весит меньше (Л3, слайд 11).
Тариф берём из подписки: у платежа своего тарифа нет.

In [ ]:
PERIOD = ("2025-12-01", "2026-07-01")   # [начало, конец): правый конец не включён

# Контрольное число строк считаем в базе ДО выгрузки: потом сверим с тем, что приехало.
expected_rows = q("""
    select count(*) from prime.payments
    where paid_at >= :a and paid_at < :b
""", a=PERIOD[0], b=PERIOD[1]).iloc[0, 0]

pay = q("""
    select p.payment_id, p.subscription_id, p.paid_at, p.amount, p.status, s.tariff_id
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where p.paid_at >= :a and p.paid_at < :b
""", a=PERIOD[0], b=PERIOD[1])

# Вопрос 1. Сколько строк и сколько ждали. inner join с подписками не должен потерять ни одной строки:
# у каждого платежа есть подписка.
print("1. строк:", len(pay), " ждали:", expected_rows, "->", "ok" if len(pay) == expected_rows else "ВНИМАНИЕ")
# Вопрос 2. Какие типы выбрал за нас pandas.
print("\n2. типы:"); print(pay.dtypes)
# Вопрос 3. Сколько весит. deep=True обязателен: без него pandas не считает сами строки текста.
print(f"\n3. вес: {pay.memory_usage(deep=True).sum() / 1e6:.1f} МБ")

**Вывод.** Приехало 1 268 997 строк, ровно столько же, сколько в базе: соединение с подписками (`join`) ничего не потеряло.
Время платежа (`paid_at`) пришло с поясом UTC, сумма (`amount`) пришла дробным числом (`float64`): для сумм по выручке
этого хватает (Л3, слайд 12). Идентификаторы (`payment_id`, `subscription_id`) пришли объектами (`object`). Таблица весит 284 МБ.

**Следующий шаг: вопросы 4-6.** Строк столько, сколько ждали, но ещё не ясно, одна ли строка это один платёж,
есть ли пропуски и невозможные значения.

In [ ]:
# Вопрос 4. Одна строка это один платёж? Тогда payment_id уникален.
print("4. payment_id уникален:", pay["payment_id"].is_unique)

# Двойные списания (С3): та же подписка и сумма меньше чем через 20 минут. Ищем среди успешных.
ok = pay[pay["status"] == "success"]
dupes = find_duplicate_payments(ok)
print(f"   двойных списаний: {len(dupes)} на сумму {dupes['amount'].sum():,.0f} руб.".replace(",", " "))

# Вопрос 5. Где пропуски.
print("\n5. пропуски по колонкам:"); print(pay.isna().sum())

# Вопрос 6. Нет ли невозможных значений.
print("\n6. статусы:"); print(pay["status"].value_counts(dropna=False))
print("   сумма <= 0:", (pay["amount"] <= 0).sum())

**Вывод.** Идентификатор платежа (`payment_id`) уникален: одна строка это одна попытка списания. Пропусков нет ни в одной колонке.
Статусы три: успешные (`success`) 1 211 707, неудачные (`failed`) 41 944, возвраты (`refunded`) 15 346. Сумм ноль и меньше нет.
Функция поиска двойных списаний (`find_duplicate_payments`) нашла 1 189 повторов на 362 961 руб.

**Следующий шаг: проверить, что это действительно двойные списания.** Если это обычные продления,
исключать их нельзя: мы бы занизили выручку. Поэтому смотрим, сколько времени проходит между платежами.

#### Почему мы считаем их двойными списаниями

Подписка списывается раз в 30 дней. Значит, между двумя успешными платежами одной подписки на одну сумму
в норме проходит около 30 дней. Если проходят минуты, это не продление, а повтор того же списания.
Проверяем это на наших данных, а не верим порогу с семинара на слово.

In [ ]:
# Для каждого успешного платежа: сколько прошло с предыдущего платежа той же подписки на ту же сумму.
s = ok.sort_values(["subscription_id", "amount", "paid_at"])
gap = s.groupby(["subscription_id", "amount"])["paid_at"].diff()   # у первого платежа подписки будет NaT

# Раскладываем промежутки по корзинам: где живут "нормальные" платежи, а где повторы.
bins = pd.to_timedelta(["0s", "1min", "20min", "1D", "25D", "35D", "3650D"])
labels = ["меньше 1 минуты", "1-20 минут", "20 минут - 1 день", "1-25 дней", "25-35 дней (продление)", "больше 35 дней"]
print("Промежуток до предыдущего такого же платежа:")
print(pd.cut(gap.dropna(), bins=bins, labels=labels, right=False).value_counts(sort=False))

# Анализ чувствительности (С3): если ответ не меняется в широком диапазоне порога,
# значит он не зависит от нашего выбора "20 минут".
print("\nПорог -> сколько повторов:")
for m in (1, 5, 15, 20, 60, 180, 1440, 7 * 1440):
    print(f"{m:>6} мин -> {(gap < pd.Timedelta(minutes=m)).sum()}")

**Вывод.** Промежутки делятся на две группы без середины: 1 189 повторов через 0-20 минут и 776 726 продлений через 25-35 дней.
Между 20 минутами и 25 днями нет ни одного платежа. Порог можно ставить от 15 минут до недели: число повторов одно и то же, 1 189.
Значит, ответ не зависит от нашего выбора порога.

**Следующий шаг: посмотреть на повторы вблизи** и проверить, вернули ли за них деньги. Если вернули,
то в выручке их и так нет; если нет, их надо убрать самим.

In [ ]:
# Как выглядят повторы вблизи: через сколько минут после первого платежа и сколько подписок задето.
is_dup = gap < pd.Timedelta(minutes=20)
print("медиана промежутка у повторов:", gap[is_dup].median())
print("самый длинный промежуток у повторов:", gap[is_dup].max())
print("подписок с повтором:", s.loc[is_dup, "subscription_id"].nunique())

# Вернули ли эти деньги? Ищем возврат (refunded) по той же подписке в течение 7 дней после повтора.
refunds = pay.loc[pay["status"] == "refunded", ["subscription_id", "paid_at"]]
d = s.loc[is_dup, ["subscription_id", "paid_at"]].merge(refunds, on="subscription_id", suffixes=("", "_ref"))
d = d[(d["paid_at_ref"] >= d["paid_at"]) & (d["paid_at_ref"] < d["paid_at"] + pd.Timedelta(days=7))]
print("повторов, по которым был возврат в течение 7 дней:", d["paid_at"].nunique(), "из", is_dup.sum())

# Пример одной подписки с повтором: все её успешные платежи в периоде.
example = s.loc[is_dup, "subscription_id"].iloc[0]
s.loc[s["subscription_id"] == example, ["paid_at", "amount", "status"]]

**Вывод.** Медиана промежутка у повторов 7 минут 51 секунда, самый длинный 14 минут 59 секунд: так выглядит повтор запроса
платёжным шлюзом, а не решение клиента. Задето 1 185 подписок. Возвратов по повторам нет ни одного: деньги взяли дважды
и не вернули. В примере подписка платит раз в месяц, а 28 апреля платит дважды с разницей 5 минут.

**Решение: исключаем повторы из выручки** (раздел 1.5). Для бизнеса это отдельная находка: 1 185 клиентам надо вернуть деньги.

**Следующий шаг: вопрос 7, дыры по дням.** От полноты дней зависит, можно ли сравнивать месяцы между собой.

In [ ]:
# Вопрос 7. Какой период покрыт и нет ли дыр по дням.
# Сравниваем дни, которые есть в данных, с полным календарём периода.
days_in_data = pd.DatetimeIndex(pay["paid_at"].dt.tz_localize(None).dt.normalize().unique())
all_days = pd.date_range(PERIOD[0], PERIOD[1], freq="D", inclusive="left")   # inclusive="left": без 1 июля
missing = all_days.difference(days_in_data)

print("7. первый и последний платёж:", pay["paid_at"].min(), "...", pay["paid_at"].max())
print("   дней в календаре:", len(all_days), " дней с платежами:", len(days_in_data))
print("   дни без единого платежа:", [d.strftime("%Y-%m-%d") for d in missing])

**Вывод.** Период покрыт с 1 декабря 2025 по 30 июня 2026: 212 дней в календаре, платежи есть в 210.
Нет ни одного платежа 14 и 15 апреля 2026: это дыра в выгрузке, известная с Л5. Апрель будем считать
в пересчёте на день только по 28 дням с данными. Февраль и март дыра не задевает.

**Следующий шаг: посмотреть, какие суммы реально списывались по месяцам.** Это ответит на два вопроса сразу:
когда новая цена дошла до клиентов и настоящая ли строка 429 руб.

### 1.3 Какие суммы реально списывались

Цены в справочнике поменялись 1 марта. Но это не значит, что 1 марта все стали платить по-новому:
новая цена могла включаться только при следующем продлении.

In [ ]:
# Месяц платежа по UTC. tz_localize(None) снимает пояс перед to_period: иначе pandas предупреждает,
# что пояс потеряется (Л4, слайд 23). Время при этом не сдвигается: остаются часы UTC.
ok = pay[pay["status"] == "success"].copy()
ok["month"] = ok["paid_at"].dt.tz_localize(None).dt.to_period("M")

# Строки: месяц. Колонки: тариф и сумма. В клетке: сколько платежей.
amounts = ok.groupby(["month", "tariff_id", "amount"]).size().unstack(["tariff_id", "amount"], fill_value=0)
amounts

**Вывод.** Новая цена включилась для всех сразу 1 марта: до марта нет ни одного платежа по 249 и 449 руб., с марта ни одного
по 199 и 399 руб. Платежей на 429 руб. нет ни одного, значит строка 429 руб. в справочнике это ошибка ввода, а не настоящая цена.

Отсюда важное следствие для метода: эффект цены можно мерить чисто на переходе февраль -> март, без переходного периода.

**Следующий шаг: подписки.** Для активной базы и оттока нужны даты начала и конца подписок и канал владельца.

### 1.4 Подписки

Канал привлечения (`acquisition_channel`) берём у **владельца** подписки. У части клиентов канала нет: считаем их
с параметром "не выбрасывать пропуски" (`dropna=False`), чтобы они не пропали молча (Л3, слайд 25).

In [ ]:
subs = q("""
    select s.subscription_id, s.tariff_id, s.started_at, s.ended_at, s.is_trial,
           c.acquisition_channel
    from prime.subscriptions s
    join prime.clients c on c.client_id = s.owner_client_id
""")

expected_subs = q("select count(*) from prime.subscriptions").iloc[0, 0]

print("строк:", len(subs), " ждали:", expected_subs, "->", "ok" if len(subs) == expected_subs else "ВНИМАНИЕ")
print("subscription_id уникален:", subs["subscription_id"].is_unique)
print("\nтипы:"); print(subs.dtypes)
print("\nпропуски:"); print(subs.isna().sum())
# Инвариант двух колонок: подписка не может закончиться раньше, чем началась.
print("\nзакончились раньше начала:", (subs["ended_at"] < subs["started_at"]).sum())
print("\nканал владельца, вместе с пропусками:")
print(subs["acquisition_channel"].value_counts(dropna=False))

**Вывод.** Подписок 333 905, ровно как в базе, идентификатор подписки (`subscription_id`) уникален.
Дата конца (`ended_at`) пуста у 167 978 подписок: это штатно, они ещё действуют. Подписок, которые закончились раньше,
чем начались, нет. Канала нет у 6 736 подписок: без параметра `dropna=False` они бы молча выпали из разбивки по каналам.

**Следующий шаг: подвести итог приёмки и собрать чистые таблицы**, чтобы дальше все расчёты шли от одних и тех же данных.

### 1.5 Итог приёмки и чистые таблицы

| Что нашли | Что делаем |
|---|---|
| Строка 429 руб. в справочнике перекрывает две соседние, платежей на 429 руб. нет ни одного | убираем строку из справочника |
| Новая цена включилась для всех 1 марта 2026: до марта нет ни одного платежа по новой цене, с марта ни одного по старой | эффект цены меряем на переходе февраль -> март |
| 1 189 двойных списаний (повтор через 0-15 минут), возвратов по ним нет | исключаем из выручки |
| 14 и 15 апреля 2026 нет ни одного платежа: дыра в выгрузке | апрель считаем в день только по дням с данными |
| Неудачные (`failed`) и возвраты (`refunded`) | в выручку не берём |
| 6 736 подписок без канала владельца | показываем строкой "без канала" |

In [ ]:
# Справочник без кривой строки 429 руб.: ни одного платежа на 429 руб. нет (раздел 1.3), значит это ошибка ввода.
tariffs_fixed = tariffs[tariffs["monthly_fee"] != 429].copy()

# Проверяем, что пересечений больше нет. Проверка, которая может упасть, и есть настоящая проверка.
a = tariffs_fixed.reset_index(names="row_id")
pairs = a.merge(a, on="tariff_id", suffixes=("_a", "_b"))
n_overlap = ((pairs["row_id_a"] < pairs["row_id_b"])
             & (pairs["valid_from_a"] <= pairs["valid_to_b"])
             & (pairs["valid_from_b"] <= pairs["valid_to_a"])).sum()
assert n_overlap == 0, "в справочнике остались пересечения"
print(tariffs_fixed[["tariff_id", "tariff_name", "monthly_fee", "valid_from", "valid_to"]].to_string(index=False))

**Вывод.** В чистом справочнике (`tariffs_fixed`) 4 строки, пересечений нет, проверка (`assert`) прошла:
199 и 399 руб. до 28.02.2026, 249 и 449 руб. с 01.03.2026. Повышение: "Базовый" +25,1%, "Расширенный" +12,5%.

**Следующий шаг: убрать двойные списания** и собрать рабочую таблицу платежей для всех следующих расчётов.

In [ ]:
success = pay[pay["status"] == "success"]   # деньги приносят только успешные платежи

# Исключили платежи, чтобы не вести двойной счёт по двойным списаниям за 1 подписку в течение 20 минут.
# (Нормально, если платёж повторяется через 25+ дней. Подтверждено данными: ячейка с корзинами промежутков выше.)
dupes = find_duplicate_payments(success, gap_minutes=20)   # функция возвращает ВТОРОЙ платёж пары, первый остаётся
paid = success[~success.index.isin(dupes.index)].copy()

# Месяц платежа по UTC: понадобится на шаге 2.
paid["month"] = paid["paid_at"].dt.tz_localize(None).dt.to_period("M")

# Контроль: убрали ровно столько строк, сколько нашли дублей, и ни одной лишней.
assert len(paid) == len(success) - len(dupes), "убрали не те строки"
print(f"успешных платежей: {len(success):>9}")
print(f"из них дублей:     {len(dupes):>9}")
print(f"осталось (paid):   {len(paid):>9}")
print(f"выручка до чистки:   {success['amount'].sum():>15,.2f} руб.".replace(",", " "))
print(f"выручка после:       {paid['amount'].sum():>15,.2f} руб.".replace(",", " "))

**Вывод.** Из 1 211 707 успешных платежей убрали 1 189 повторов, в рабочей таблице (`paid`) осталось 1 210 518.
Выручка за 7 месяцев: 368 919 493 руб. до чистки и 368 556 532 руб. после, разница 362 961 руб.
Проверка (`assert`) подтвердила, что убрали ровно повторы и ни одной лишней строки.

**Следующий шаг: метрики по месяцам.** Данные чистые, теперь можно собирать таблицу, из которой будем раскладывать выручку.

## 2. Метрики по месяцам и зерно расчёта

**Зерно рабочей таблицы: одна строка это месяц × тариф × канал владельца.** Почему именно так:

- **месяц**: подписка списывается раз в 30 дней, значит месяц это естественный период, в котором каждая живая подписка платит примерно один раз;
- **тариф**: цену меняли по тарифам (199 -> 249 и 399 -> 449), и эффект цены видно только внутри тарифа;
- **канал**: чтобы ответить "где" (аддитивная модель, Л6): не ушли ли после повышения клиенты одного канала.

Мельче (платёж или подписка) не нужно: вопрос про выручку компании, а не про отдельного клиента. Крупнее (просто месяц) нельзя:
пропадут ответы "где" и эффект структуры тарифов.

**Как считаем каждую колонку:**

| Колонка | Определение |
|---|---|
| активные подписки (`active`) | подписки, активные в месяце: начались до конца месяца и не закончились до его начала (как в С6) |
| число платежей (`n_payments`) | успешные платежи без двойных списаний (таблица `paid`) |
| выручка (`revenue`) | сумма этих платежей, руб. |
| дней с данными (`days`) | сколько дней месяца есть в данных: у апреля 28 из 30 из-за дыры 14-15.04 |

In [ ]:
MONTHS = pd.period_range("2025-12", "2026-06", freq="M")   # 7 месяцев: три до повышения, март, три после

# Канал у подписки это канал её владельца. Пропуск заменяем на "без канала":
# иначе groupby и pivot_table молча выбросят эти строки (Л3, слайд 25; С6, раздел 2).
subs["channel"] = subs["acquisition_channel"].fillna("без канала")

# Приписываем каждому платежу канал его подписки.
# Кратность m:1: у подписки много платежей, у платежа ровно одна подписка. Если это не так, merge упадёт.
rows_before = len(paid)
paid = paid.merge(subs[["subscription_id", "channel"]], on="subscription_id", how="left", validate="m:1")
assert len(paid) == rows_before, "соединение размножило платежи"
assert paid["channel"].notna().all(), "у части платежей не нашлась подписка"
print("платежей:", len(paid), "| каналы:", sorted(paid["channel"].unique()))

**Вывод.** Каждому из 1 210 518 платежей приписан канал владельца, соединение с проверкой кратности (`validate="m:1"`)
не упало и не размножило строки. Каналов 7: шесть настоящих и "без канала".

**Следующий шаг: посчитать активные подписки, платежи и выручку в зерне месяц × тариф × канал** и сверить деньги
с рабочей таблицей: если сборка потеряет или добавит рубли, проверка упадёт.

In [ ]:
# Активные подписки по месяцам. Цикл по 7 месяцам, внутри каждого векторная маска: это быстро.
parts = []
for m in MONTHS:
    start = m.start_time.tz_localize("UTC")          # 1-е число месяца, 00:00 UTC
    end = (m + 1).start_time.tz_localize("UTC")      # 1-е число следующего месяца: правый конец не включён
    # Активна в месяце: началась раньше конца месяца И (не закончилась ИЛИ закончилась не раньше его начала).
    alive = subs[(subs["started_at"] < end) & (subs["ended_at"].isna() | (subs["ended_at"] >= start))]
    g = alive.groupby(["tariff_id", "channel"]).size().rename("active").reset_index()
    g["month"] = m
    parts.append(g)
active = pd.concat(parts, ignore_index=True)

# Платежи и выручка в том же зерне: месяц × тариф × канал.
rev = (paid.groupby(["month", "tariff_id", "channel"], as_index=False)
           .agg(n_payments=("payment_id", "size"), revenue=("amount", "sum")))

# Соединяем 1:1: в обеих таблицах одна строка на месяц × тариф × канал.
# outer + indicator показывают, есть ли платежи у клеток без активных подписок, и наоборот.
monthly = active.merge(rev, on=["month", "tariff_id", "channel"], how="outer", validate="1:1", indicator=True)
print(monthly["_merge"].value_counts())
monthly = monthly.drop(columns="_merge").fillna({"active": 0, "n_payments": 0, "revenue": 0})

# ДЕНЬГИ: выручка в витрине обязана совпасть с суммой paid до копейки.
assert abs(monthly["revenue"].sum() - paid["amount"].sum()) < 0.01, "деньги потерялись при сборке"
monthly["tariff"] = monthly["tariff_id"].map({1: "Базовый", 2: "Расширенный"})
print("строк в monthly:", len(monthly), "(7 месяцев × 2 тарифа × 7 каналов = 98)")
monthly.head()

**Вывод.** В таблице (`monthly`) 98 строк: 7 месяцев × 2 тарифа × 7 каналов. Все 98 клеток нашли пару
(`both`): нет клеток с платежами, но без активных подписок, и наоборот. Выручка сошлась с рабочей таблицей до копейки.

**Следующий шаг: свести таблицу по месяцам и разложить выручку на три множителя в пересчёте на день.**
Месяцы разной длины (февраль 28 дней) и апрель с дырой иначе сравнивать нельзя (Л5, слайд 18).

In [ ]:
# Сводка по месяцам. Сколько дней каждого месяца есть в данных: у апреля дыра в 2 дня.
paid["day"] = paid["paid_at"].dt.floor("D")
days_with_data = paid.groupby("month")["day"].nunique()

m = monthly.groupby("month")[["active", "n_payments", "revenue"]].sum()
m["days"] = days_with_data
m["days_in_month"] = m.index.days_in_month               # сколько дней в календарном месяце (Л5, слайд 18)

# Средняя выручка в день = активные подписки × среднее число платежей на подписку в день × средняя цена платежа.
# В пересчёте на день месяцы разной длины становятся сравнимы, а апрель считается только по дням с данными.
m["revenue_per_day"] = m["revenue"] / m["days"]                # средняя выручка в день, руб.
m["per_sub_day"] = m["n_payments"] / m["active"] / m["days"]   # среднее число платежей на одну активную подписку в день
m["price"] = m["revenue"] / m["n_payments"]                    # средняя цена одного платежа, руб.

# Проверка, что три множителя дают среднюю выручку в день точно: модель обязана сходиться.
assert np.allclose(m["active"] * m["per_sub_day"] * m["price"], m["revenue_per_day"])

m["growth_per_day_pct"] = m["revenue_per_day"].pct_change() * 100   # прирост средней выручки в день к прошлому месяцу, %

# Для показа: среднее число платежей на 1000 активных подписок в день. Это то же per_sub_day, умноженное на 1000:
# иначе при двух знаках после запятой все месяцы выглядят одинаково (0.03).
m.assign(per_1000_subs_day=m["per_sub_day"] * 1000).drop(columns="per_sub_day")

**Вывод.**

| Показатель | Февраль 2026 | Март 2026 | Изменение |
|---|---:|---:|---:|
| средняя выручка в день (`revenue_per_day`) | 1 551 975 руб. | 1 876 423 руб. | **+20,9%** |
| активные подписки (`active`) | 178 682 | 183 751 | +2,8% |
| среднее число платежей на 1000 подписок в день (`per_1000_subs_day`) | 31,60 | 31,44 | -0,5% |
| средняя цена платежа (`price`) | 274,85 руб. | 324,84 руб. | +18,2% |

- Средняя выручка в день выросла на 20,9%, почти всё дала средняя цена платежа.
- База активных подписок росла и до повышения (+2,9% и +2,0% в месяц), и в марте (+2,8%): обвала нет.
- Среднее число платежей на подписку в день почти не изменилось. Февраль выделяется из ряда (в остальных месяцах от 31,41 до 31,52):
  похоже, это особенность короткого месяца в определении "активна в месяце", а не реакция клиентов.
- После марта средняя выручка в день растёт на 2,1-2,3% в месяц, до повышения росла на 2,7-2,9%. Это сигнал, который надо проверить на оттоке.

**Следующий шаг: проверить структуру тарифов.** Средняя цена платежа может расти и без изменения цен, если клиенты
перешли на дорогой тариф (Л6, слайд 15). Тогда рост цены в таблице выше был бы не только эффектом повышения.

In [ ]:
# То же по тарифам: средняя цена платежа внутри тарифа и доля тарифа в платежах.
# Средняя цена смешивает цену и структуру (Л6, слайд 15): если доля дорогого тарифа выросла,
# средний чек растёт и без изменения цен. Поэтому смотрим долю отдельно.
t = monthly.pivot_table(index="month", columns="tariff", values=["n_payments", "revenue", "active"], aggfunc="sum")
by_tariff = pd.DataFrame({
    "price_basic": t["revenue"]["Базовый"] / t["n_payments"]["Базовый"],
    "price_ext": t["revenue"]["Расширенный"] / t["n_payments"]["Расширенный"],
    "share_ext_pct": t["n_payments"]["Расширенный"] / t["n_payments"].sum(axis=1) * 100,   # доля "Расширенного" в платежах, %
    "active_basic": t["active"]["Базовый"],
    "active_ext": t["active"]["Расширенный"],
})
by_tariff

**Вывод.** Средняя цена платежа внутри тарифа ровно равна цене из справочника: 199 -> 249 руб. у "Базового" (`price_basic`)
и 399 -> 449 руб. у "Расширенного" (`price_ext`). Доля "Расширенного" в платежах (`share_ext_pct`) держится на 37,9-38,0%
все 7 месяцев, а активные подписки обоих тарифов (`active_basic`, `active_ext`) растут. Массового перехода на дешёвый тариф нет,
значит эффект структуры почти нулевой и рост средней цены платежа это действительно повышение цен.

**Следующий шаг: факторное разложение февраль -> март.** Теперь известны все три множителя, и можно точно сказать,
сколько рублей в день дал каждый из них, со сверкой суммы влияний (`assert`).

## 3. Факторное разложение выручки

Раскладываем изменение **средней выручки в день** с февраля на март 2026: это переход, на котором поменялись цены.
Считаем в рублях в день, потому что в феврале 28 дней, а в марте 31, и сравнивать месячные суммы нельзя (Л5, слайд 18).

Порядок как на Л6:

1. **Где** изменилась выручка: аддитивная модель, складываем части по тарифам и каналам.
2. **За счёт чего**: мультипликативная модель, средняя выручка в день = активные подписки × среднее число платежей на подписку в день × средняя цена платежа.
3. **Цена или структура**: средняя цена платежа могла вырасти и от смены тарифов, отделяем это.
4. **Сочетание моделей**: по каналам сложением, внутри канала умножением (Л6, слайд 14).
5. **Контроль**: те же расчёты на соседних месяцах, где цены не менялись.

Сумма влияний в каждом расчёте обязана равняться изменению итога: это проверяет `assert`.

### 3.1 Где: тарифы и каналы

In [ ]:
FEB, MAR = pd.Period("2026-02", "M"), pd.Period("2026-03", "M")

# Средняя выручка в день для каждой клетки месяц × тариф × канал: выручку клетки делим на дни месяца с данными.
cells = monthly[monthly["month"].isin([FEB, MAR])].copy()
cells["revenue_per_day"] = cells["revenue"] / cells["month"].map(days_with_data)

total_delta = m.loc[MAR, "revenue_per_day"] - m.loc[FEB, "revenue_per_day"]   # изменение итога, руб. в день


def where(by: str) -> pd.DataFrame:
    """Аддитивная модель: изменение средней выручки в день по частям (тарифам или каналам)."""
    t = cells.pivot_table(index=by, columns="month", values="revenue_per_day", aggfunc="sum")  # sum: внутри части складываем клетки
    t.columns = ["feb", "mar"]
    t["delta"] = t["mar"] - t["feb"]                     # влияние части на итог, руб. в день
    t["delta_pct"] = t["delta"] / t["feb"] * 100         # рост части, %
    t["share_of_delta_pct"] = t["delta"] / total_delta * 100   # какую долю итогового роста дала часть, %
    # Сумма частей равна целому: иначе мы что-то потеряли (как 11 357 руб. без канала в С6).
    assert abs(t["delta"].sum() - total_delta) < 0.01, "части не сходятся с итогом"
    return t.sort_values("delta", ascending=False)


print(f"изменение средней выручки в день: {total_delta:,.0f} руб.".replace(",", " "))
where("tariff")

**Вывод.** Средняя выручка в день выросла на 324 449 руб. (+20,9%). "Базовый" дал +195 466 руб. в день (+28,0%),
"Расширенный" +128 982 руб. в день (+15,1%). Сумма частей сходится с итогом (`assert`).
Рост тарифов чуть больше роста их цен (+25,1% и +12,5%): сверху добавился рост базы подписок.

**Следующий шаг: та же разбивка по каналам.** Так проверяем гипотезу 3: не ушли ли после повышения клиенты какого-то одного канала.

In [ ]:
where("channel")

**Вывод.** Все 7 каналов выросли почти одинаково: от +20,2% (`social`) до +22,4% ("без канала").
Ни один канал не упал, и доля каждого канала в росте (`share_of_delta_pct`) близка к его доле в выручке.
Гипотеза 3 в части каналов **не подтверждается**: после повышения не просел ни один канал привлечения.

**Следующий шаг: мультипликативная модель.** Аддитивная ответила "где": везде понемногу. Теперь нужно понять,
"за счёт чего": база, частота платежей или цена.

### 3.2 За счёт чего: три множителя

In [ ]:
FACTORS = ["active", "per_sub_day", "price"]     # порядок курса: сначала "сколько", потом "почём"
LABELS = {"active": "активные подписки", "per_sub_day": "платежей на подписку в день", "price": "средняя цена платежа"}

x0 = m.loc[FEB, FACTORS].to_dict()   # значения факторов в феврале
x1 = m.loc[MAR, FACTORS].to_dict()   # значения факторов в марте

# Цепные подстановки: меняем факторы по одному в порядке FACTORS (функция contrib из src/prime/factors.py).
effects = pd.Series(contrib(x0, x1, FACTORS))
assert abs(effects.sum() - total_delta) < 0.01, "сумма влияний не равна изменению средней выручки в день"

out = pd.DataFrame({
    "было": pd.Series(x0),
    "стало": pd.Series(x1),
    "изменение_pct": (pd.Series(x1) / pd.Series(x0) - 1) * 100,
    "влияние_руб_в_день": effects,
    "доля_роста_pct": effects / total_delta * 100,
}).rename(index=LABELS)
# Для показа: платежи на подписку в день умножаем на 1000, иначе при двух знаках все значения выглядят как 0.03.
out.loc[LABELS["per_sub_day"], ["было", "стало"]] *= 1000
out.rename(index={LABELS["per_sub_day"]: "платежей на 1000 подписок в день"})

**Вывод.** Из +324 449 руб. в день:

- средняя цена платежа (`price`) дала **+288 723 руб. в день, 89,0% роста**;
- активные подписки (`active`) дали +44 028 руб. в день, 13,6%;
- среднее число платежей на подписку в день (`per_sub_day`) дало -8 302 руб. в день, -2,6%.

Сумма трёх влияний равна изменению итога (`assert`). Повышение цен приносит около 289 тыс. руб. в день,
это примерно 8,7 млн руб. за 30 дней. Гипотеза 1 **подтверждается**: рост в марте дала цена, а не база.

**Следующий шаг: проверить, не зависит ли вывод от порядка подстановок.** Совместный член достаётся фактору,
который меняют позже (Л6, слайды 10-11), и при другом порядке числа будут другими.

In [ ]:
import itertools

# Порядок подстановок меняет ответ: совместный член достаётся фактору, который меняют позже (Л6, слайды 10-11).
# Проверяем все 6 порядков. Если вывод одинаковый во всех, он не зависит от нашего выбора порядка.
orders = pd.DataFrame({" -> ".join(o): contrib(x0, x1, list(o)) for o in itertools.permutations(FACTORS)}).T
orders = orders[FACTORS].rename(columns=LABELS)
orders.loc["среднее по 6 порядкам"] = orders.mean()     # среднее по порядкам от порядка не зависит (вектор Шепли, С6)
orders.round(0)

**Вывод.** Во всех 6 порядках вывод один и тот же. Влияние средней цены платежа от 280 758 до 290 233 руб. в день,
активных подписок от 43 799 до 52 034, среднего числа платежей на подписку от -9 812 до -8 073.
Разброс не больше 9 500 руб. в день: это совместные члены. Среднее по 6 порядкам: цена +285 481, активные +47 903,
платежи на подписку -8 935 руб. в день. В записке используем порядок курса: сначала количество, потом цена.

**Следующий шаг: отделить цену от структуры тарифов.** Средняя цена платежа могла вырасти и потому,
что клиенты перешли на дорогой тариф (Л6, слайд 15).

### 3.3 Цена или структура тарифов

In [ ]:
# Средняя цена платежа = сумма по тарифам (доля тарифа в платежах × цена тарифа).
# Её изменение раскладываем на две части (Л6, слайд 15):
#   структура: доли тарифов поменялись, цены ещё старые;
#   цена: цены поменялись при новых долях.
tp = cells.pivot_table(index="tariff", columns="month", values=["n_payments", "revenue"], aggfunc="sum")
share0 = tp["n_payments"][FEB] / tp["n_payments"][FEB].sum()   # доли тарифов в платежах, февраль
share1 = tp["n_payments"][MAR] / tp["n_payments"][MAR].sum()   # доли тарифов в платежах, март
p0 = tp["revenue"][FEB] / tp["n_payments"][FEB]                # средняя цена платежа тарифа, февраль
p1 = tp["revenue"][MAR] / tp["n_payments"][MAR]                # средняя цена платежа тарифа, март

structure = ((share1 - share0) * p0).sum()     # руб. на платёж за счёт смены долей
pure_price = (share1 * (p1 - p0)).sum()        # руб. на платёж за счёт самих цен
assert abs(structure + pure_price - (x1["price"] - x0["price"])) < 1e-9, "цена и структура не сходятся"

# В рубли в день: в contrib цена стоит последней, её влияние = активные(март) × платежи на подписку(март) × изменение цены.
scale = x1["active"] * x1["per_sub_day"]       # среднее число платежей в день в марте
split = pd.Series({"структура тарифов": structure * scale, "цены тарифов": pure_price * scale})
assert abs(split.sum() - effects["price"]) < 0.01, "не сходится с влиянием цены из 3.2"

print(pd.DataFrame({"доля_февраль_pct": share0 * 100, "доля_март_pct": share1 * 100,
                    "цена_февраль": p0, "цена_март": p1}).round(3))
print()
print("влияние, руб. в день:")
print(split.round(0))

**Вывод.** Доля "Расширенного" в платежах почти не изменилась: 37,93% в феврале и 37,92% в марте.
Структура тарифов дала -103 руб. в день, сами цены тарифов +288 826 руб. в день. Вместе это ровно влияние
средней цены платежа из 3.2 (`assert`). Клиенты не стали переходить на дешёвый "Базовый":
гипотеза 3 в части тарифов тоже **не подтверждается**.

**Следующий шаг: сочетание моделей**, канал × фактор. Проверяем, что картина "цена дала почти всё" держится в каждом канале,
а не только в среднем.

### 3.4 Сочетание моделей: канал × фактор

In [ ]:
# Внутри каждого канала раскладываем его изменение на те же три множителя (Л6, слайд 14).
ch = cells.groupby(["channel", "month"])[["active", "n_payments", "revenue"]].sum()
ch["days"] = ch.index.get_level_values("month").map(days_with_data)
ch["per_sub_day"] = ch["n_payments"] / ch["active"] / ch["days"]
ch["price"] = ch["revenue"] / ch["n_payments"]

rows = {}
for c in ch.index.get_level_values("channel").unique():
    rows[c] = contrib(ch.loc[(c, FEB), FACTORS].to_dict(), ch.loc[(c, MAR), FACTORS].to_dict(), FACTORS)
by_ch_factor = pd.DataFrame(rows).T[FACTORS]
by_ch_factor["всего"] = by_ch_factor.sum(axis=1)
by_ch_factor = by_ch_factor.sort_values("всего", ascending=False)
by_ch_factor.loc["итого"] = by_ch_factor.sum()

# Сумма всех клеток равна изменению итога.
assert abs(by_ch_factor.loc["итого", "всего"] - total_delta) < 0.01, "каналы × факторы не сходятся с итогом"
by_ch_factor.rename(columns=LABELS).round(0)

**Вывод.** В каждом канале главный фактор это средняя цена платежа: от +5 766 руб. в день ("без канала") до +81 275 (`organic`).
Активные подписки выросли во всех каналах. Среднее число платежей на подписку чуть снизилось почти везде, кроме "без канала".
Сумма всех клеток равна изменению итога, 324 449 руб. в день (`assert`). Итоги по факторам чуть отличаются от 3.2
(например, 44 042 против 44 028 у активных): внутри каналов свои средние, а сумма всего та же.

**Следующий шаг: контроль на соседних месяцах.** Если эффект цены настоящий, на переходах без смены цен
влияние средней цены платежа должно быть около нуля.

### 3.5 Контроль: все переходы месяц к месяцу

In [ ]:
# Если эффект цены настоящий, он должен быть только на переходе февраль -> март,
# а на соседних переходах (цены не менялись) влияние средней цены платежа должно быть около нуля.
steps = {}
for a, b in zip(m.index[:-1], m.index[1:]):
    eff = contrib(m.loc[a, FACTORS].to_dict(), m.loc[b, FACTORS].to_dict(), FACTORS)
    assert abs(sum(eff.values()) - (m.loc[b, "revenue_per_day"] - m.loc[a, "revenue_per_day"])) < 0.01
    steps[f"{a} -> {b}"] = eff
all_steps = pd.DataFrame(steps).T[FACTORS]
all_steps["всего"] = all_steps.sum(axis=1)
all_steps.rename(columns=LABELS).round(0)

**Вывод.**

- На переходах без смены цен влияние средней цены платежа от -422 до +781 руб. в день, почти ноль. На февраль -> март +288 723.
  Значит, рост в марте это действительно эффект повышения цен, а не обычная сезонность.
- Активные подписки каждый месяц добавляют примерно одинаково, 31-44 тыс. руб. в день, и после повышения тоже (38-42 тыс.).
- **В рублях рост после повышения не замедлился:** +42 413, +39 591, +44 736 руб. в день за апрель-июнь против +42 994 и +40 179 до марта.
  Процент роста стал меньше (раздел 2) только потому, что сама выручка стала на 21% больше. Тревожный сигнал из раздела 2 снимается.
- Среднее число платежей на подписку прыгает от -8 302 до +9 515 руб. в день без связи с ценой: это длина месяцев и определение "активна в месяце".

**Следующий шаг: отток (шаг 4).** Рост базы в рублях держится, но он может скрывать одновременный рост и новых подписок, и оттока.
Гипотезу 2 проверяем напрямую: долей подписок, которые закончились в месяце.

## 4. Проверка гипотезы про отток

*Заполним на следующих шагах.*

## 5. Графики

*Заполним на следующих шагах.*

## 6. Вывод

*Заполним на следующих шагах.*